In [1]:
import os
import subprocess
from dotenv import load_dotenv
from pathlib import Path

from collections import Counter

import pandas as pd

import deeparg
from Bio import SeqIO

from pprint import pprint
from tqdm.auto import tqdm

load_dotenv()

True

In [2]:
INPUT_DIR = "/home/ubuntu/projects/biodata/DNA-BERT/data/raw/Wetland Metagenome Assembled Genomes"

In [3]:
file_extension_counts = Counter(os.path.splitext(f)[1].lower().lstrip(".") for root, dirs, files in os.walk(INPUT_DIR) for f in files)

pprint(file_extension_counts)

Counter({'faa': 641,
         'fna': 641,
         'gff': 640,
         'log': 640,
         'txt': 640,
         'tbl': 640,
         'fsa': 640,
         'gbk': 640,
         'ffn': 640,
         'sqn': 640,
         'tsv': 640,
         'err': 640})


| Extension | Meaning                  | Contains                                                                                                       |
| --------- | ------------------------ | -------------------------------------------------------------------------------------------------------------- |
| **.fna**  | FASTA Nucleotide         | Whole genome nucleotide sequences (assembled contigs/chromosomes/scaffolds)                                    |
| **.faa**  | FASTA Amino Acid         | Predicted protein sequences translated from CDS features                                                       |
| **.ffn**  | FASTA Feature Nucleotide | Nucleotide sequences of genes/CDS only (not the entire genome)                                                 |
| **.fsa**  | FASTA Sequence           | Generic FASTA file. Often used during submission workflows. Can contain nucleotide sequences similar to `.fna` |
| **.gff**  | General Feature Format   | Genome annotations: genes, CDS, tRNA, rRNA, coordinates, strands, attributes                                   |
| **.gbk**  | GenBank Format           | Sequence + annotations + metadata in one rich file                                                             |
| **.tbl**  | Feature Table            | NCBI annotation table used during genome submission                                                            |
| **.sqn**  | Sequin File              | Binary submission package used by NCBI's Sequin/BankIt submission system                                       |
| **.err**  | Error Log                | Validation or submission errors generated by annotation/submission software                                    |
| **.log**  | Log File                 | Program execution logs                                                                                         |
| **.tsv**  | Tab-Separated Values     | Annotation summaries, statistics, metadata tables                                                              |
| **.txt**  | Plain Text               | Notes, reports, readme files, miscellaneous output                                                             |


#### Relationship between the important biological files

<pre>
Genome assembly 
    |
    +-- genome.fna   (full nucleotide sequence)
    |
    +-- annotations.gff
    |
    +-- proteins.faa
    |
    +-- genes.ffn
    |
    +-- genome.gbk
</pre>


In [4]:
faa_files = [os.path.join(root, f) for root, dirs, files in os.walk(INPUT_DIR) for f in files if f.endswith(".faa")]
print("Unique .faa file types:", len(faa_files))

Unique .faa file types: 641


In [5]:
OUTPUT_DIR = "/home/ubuntu/projects/biodata/DNA-BERT/data/interim/Wetland Metagenome Assembled Genomes/"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

In [6]:
filed = 0
filed_files = []
completed = 0
total = len(faa_files)

for faa_file in tqdm(faa_files):
    sample = os.path.splitext(os.path.basename(faa_file))[0]
    output_prefix = os.path.join(OUTPUT_DIR, sample)

    cmd = [
        "deeparg",
        "predict",
        "--model",
        "LS",
        "--type",
        "prot",
        "--input",
        faa_file,
        "--output",
        output_prefix,
    ]

    print("Running: " + " ".join(cmd))
    exit_code = subprocess.call(cmd)

    if exit_code != 0:
        filed += 1
        filed_files.append(faa_file)
        print("FAILED: " + faa_file)
    else:
        completed += 1
        print("COMPLETED: " + faa_file)
    print(f"Total: {total}, Completed: {completed}, Failed: {filed}")


  0%|          | 0/641 [00:00<?, ?it/s]

Running: deeparg predict --model LS --type prot --input /home/ubuntu/projects/biodata/DNA-BERT/data/raw/Wetland Metagenome Assembled Genomes/S1A9/bin.1/bin.1.faa --output /home/ubuntu/projects/biodata/DNA-BERT/data/interim/Wetland Metagenome Assembled Genomes/bin.1


INFO:httpx:HTTP Request: HEAD https://huggingface.co/gaarangoa/deeparg/resolve/main/bin/linux-x86_64/diamond "HTTP/1.1 302 Found"
INFO:root:Using DIAMOND: /home/ubuntu/.cache/huggingface/hub/models--gaarangoa--deeparg/snapshots/07d2eaa78502f41725c4222de1288b8b2f0d6259/bin/linux-x86_64/diamond (diamond version 2.1.24)
INFO:httpx:HTTP Request: HEAD https://huggingface.co/gaarangoa/deeparg/resolve/main/LS/database/features.dmnd "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/gaarangoa/deeparg/resolve/main/LS/database/features.gene.length "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/gaarangoa/deeparg/07d2eaa78502f41725c4222de1288b8b2f0d6259/LS%2Fdatabase%2Ffeatures.gene.length "HTTP/1.1 200 OK"
INFO:root:DIAMOND blastp alignment
INFO:root:Running: /home/ubuntu/.cache/huggingface/hub/models--gaarangoa--deeparg/snapshots/07d2eaa78502f41725c4222de1288b8b2f0d6259/bin/linux-x86_64/diamond blastp -q '/ho

COMPLETED: /home/ubuntu/projects/biodata/DNA-BERT/data/raw/Wetland Metagenome Assembled Genomes/S1A9/bin.1/bin.1.faa
Total: 641, Completed: 1, Failed: 0
Running: deeparg predict --model LS --type prot --input /home/ubuntu/projects/biodata/DNA-BERT/data/raw/Wetland Metagenome Assembled Genomes/S1A9/bin.13/bin.13.faa --output /home/ubuntu/projects/biodata/DNA-BERT/data/interim/Wetland Metagenome Assembled Genomes/bin.13


INFO:httpx:HTTP Request: HEAD https://huggingface.co/gaarangoa/deeparg/resolve/main/bin/linux-x86_64/diamond "HTTP/1.1 302 Found"
INFO:root:Using DIAMOND: /home/ubuntu/.cache/huggingface/hub/models--gaarangoa--deeparg/snapshots/07d2eaa78502f41725c4222de1288b8b2f0d6259/bin/linux-x86_64/diamond (diamond version 2.1.24)
INFO:httpx:HTTP Request: HEAD https://huggingface.co/gaarangoa/deeparg/resolve/main/LS/database/features.dmnd "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/gaarangoa/deeparg/resolve/main/LS/database/features.gene.length "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/gaarangoa/deeparg/07d2eaa78502f41725c4222de1288b8b2f0d6259/LS%2Fdatabase%2Ffeatures.gene.length "HTTP/1.1 200 OK"
INFO:root:DIAMOND blastp alignment
INFO:root:Running: /home/ubuntu/.cache/huggingface/hub/models--gaarangoa--deeparg/snapshots/07d2eaa78502f41725c4222de1288b8b2f0d6259/bin/linux-x86_64/diamond blastp -q '/ho

...

COMPLETED: /home/ubuntu/projects/biodata/DNA-BERT/data/raw/Wetland Metagenome Assembled Genomes/S1H12/bin.12/bin.12.faa
Total: 641, Completed: 638, Failed: 1
Running: deeparg predict --model LS --type prot --input /home/ubuntu/projects/biodata/DNA-BERT/data/raw/Wetland Metagenome Assembled Genomes/S1H12/bin.24/bin.24.faa --output /home/ubuntu/projects/biodata/DNA-BERT/data/interim/Wetland Metagenome Assembled Genomes/bin.24


INFO:httpx:HTTP Request: HEAD https://huggingface.co/gaarangoa/deeparg/resolve/main/bin/linux-x86_64/diamond "HTTP/1.1 302 Found"
INFO:root:Using DIAMOND: /home/ubuntu/.cache/huggingface/hub/models--gaarangoa--deeparg/snapshots/07d2eaa78502f41725c4222de1288b8b2f0d6259/bin/linux-x86_64/diamond (diamond version 2.1.24)
INFO:httpx:HTTP Request: HEAD https://huggingface.co/gaarangoa/deeparg/resolve/main/LS/database/features.dmnd "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/gaarangoa/deeparg/resolve/main/LS/database/features.gene.length "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/gaarangoa/deeparg/07d2eaa78502f41725c4222de1288b8b2f0d6259/LS%2Fdatabase%2Ffeatures.gene.length "HTTP/1.1 200 OK"
INFO:root:DIAMOND blastp alignment
INFO:root:Running: /home/ubuntu/.cache/huggingface/hub/models--gaarangoa--deeparg/snapshots/07d2eaa78502f41725c4222de1288b8b2f0d6259/bin/linux-x86_64/diamond blastp -q '/ho

COMPLETED: /home/ubuntu/projects/biodata/DNA-BERT/data/raw/Wetland Metagenome Assembled Genomes/S1H12/bin.24/bin.24.faa
Total: 641, Completed: 639, Failed: 1
Running: deeparg predict --model LS --type prot --input /home/ubuntu/projects/biodata/DNA-BERT/data/raw/Wetland Metagenome Assembled Genomes/S1H12/bin.14/bin.14.faa --output /home/ubuntu/projects/biodata/DNA-BERT/data/interim/Wetland Metagenome Assembled Genomes/bin.14


INFO:httpx:HTTP Request: HEAD https://huggingface.co/gaarangoa/deeparg/resolve/main/bin/linux-x86_64/diamond "HTTP/1.1 302 Found"
INFO:root:Using DIAMOND: /home/ubuntu/.cache/huggingface/hub/models--gaarangoa--deeparg/snapshots/07d2eaa78502f41725c4222de1288b8b2f0d6259/bin/linux-x86_64/diamond (diamond version 2.1.24)
INFO:httpx:HTTP Request: HEAD https://huggingface.co/gaarangoa/deeparg/resolve/main/LS/database/features.dmnd "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/gaarangoa/deeparg/resolve/main/LS/database/features.gene.length "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/gaarangoa/deeparg/07d2eaa78502f41725c4222de1288b8b2f0d6259/LS%2Fdatabase%2Ffeatures.gene.length "HTTP/1.1 200 OK"
INFO:root:DIAMOND blastp alignment
INFO:root:Running: /home/ubuntu/.cache/huggingface/hub/models--gaarangoa--deeparg/snapshots/07d2eaa78502f41725c4222de1288b8b2f0d6259/bin/linux-x86_64/diamond blastp -q '/ho

COMPLETED: /home/ubuntu/projects/biodata/DNA-BERT/data/raw/Wetland Metagenome Assembled Genomes/S1H12/bin.14/bin.14.faa
Total: 641, Completed: 640, Failed: 1


In [7]:
print(f"Processing completed. Total: {total}, Completed: {completed}, Failed: {filed}")
if filed_files:
    print("Failed files:")
    for f in filed_files:
        print(f)

Processing completed. Total: 641, Completed: 640, Failed: 1
Failed files:
/home/ubuntu/projects/biodata/DNA-BERT/data/raw/Wetland Metagenome Assembled Genomes/S1A1/bin.3/._bin.3.faa


In [13]:
OUTPUT_DIR = Path(OUTPUT_DIR)
SUMMARY_DIR = OUTPUT_DIR / "_summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

In [14]:
def read_deeparg_file(path):
    if path.stat().st_size == 0:
        return pd.DataFrame()

    try:
        return pd.read_csv(path, sep="\t")
    except Exception:
        return pd.read_csv(path, sep="\t", engine="python")

In [15]:
def find_col(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None

In [16]:
rows = []
all_arg_rows = []

arg_files = sorted(OUTPUT_DIR.rglob("*.mapping.ARG"))

for arg_file in arg_files:
    sample = arg_file.name.replace(".mapping.ARG", "")
    potential_file = arg_file.with_name(sample + ".mapping.potential.ARG")
    align_file = arg_file.with_name(sample + ".align.daa.tsv")

    arg_df = read_deeparg_file(arg_file)
    potential_df = read_deeparg_file(potential_file) if potential_file.exists() else pd.DataFrame()

    class_col = find_col(arg_df, ["predicted_ARG-class", "predicted_arg-class", "arg_class", "ARG-class"])

    gene_col = find_col(arg_df, ["read_id", "query", "qseqid", "sequence", "gene", "ORF_ID"])

    best_hit_col = find_col(arg_df, ["best-hit", "best_hit", "sseqid", "ARG", "arg"])

    prob_col = find_col(arg_df, ["probability", "prob", "score"])

    n_args = len(arg_df)
    n_potential_args = len(potential_df)

    if class_col and n_args > 0:
        class_counts = arg_df[class_col].value_counts()
        top_class = class_counts.index[0]
        top_class_count = int(class_counts.iloc[0])
        n_classes = int(arg_df[class_col].nunique())
    else:
        top_class = None
        top_class_count = 0
        n_classes = 0

    rows.append(
        {
            "sample": sample,
            "has_ARG": n_args > 0,
            "n_ARGs": n_args,
            "n_potential_ARGs": n_potential_args,
            "n_ARG_classes": n_classes,
            "top_ARG_class": top_class,
            "top_ARG_class_count": top_class_count,
            "arg_file": str(arg_file),
            "potential_arg_file_exists": potential_file.exists(),
            "align_file_exists": align_file.exists(),
        }
    )

    if n_args > 0:
        tmp = arg_df.copy()
        tmp["sample"] = sample
        tmp["arg_file"] = str(arg_file)

        if class_col:
            tmp["deeparg_label"] = tmp[class_col]
        else:
            tmp["deeparg_label"] = "ARG"

        if gene_col:
            tmp["sequence_id"] = tmp[gene_col]
        else:
            tmp["sequence_id"] = tmp.index.astype(str)

        if best_hit_col:
            tmp["deeparg_best_hit"] = tmp[best_hit_col]
        else:
            tmp["deeparg_best_hit"] = None

        if prob_col:
            tmp["deeparg_score"] = tmp[prob_col]
        else:
            tmp["deeparg_score"] = None

        all_arg_rows.append(tmp)


summary_df = pd.DataFrame(rows)

if all_arg_rows:
    all_args_df = pd.concat(all_arg_rows, ignore_index=True)
else:
    all_args_df = pd.DataFrame()


summary_df.to_csv(SUMMARY_DIR / "deeparg_file_summary.tsv", sep="\t", index=False)
all_args_df.to_csv(SUMMARY_DIR / "deeparg_all_positive_ARGs.tsv", sep="\t", index=False)


In [17]:
# Confusion-matrix-style table:
# rows = genome/bin/sample
# columns = ARG class
# values = count of ARGs in that sample/class
if not all_args_df.empty and "deeparg_label" in all_args_df.columns:
    class_matrix = pd.pivot_table(all_args_df, index="sample", columns="deeparg_label", values="sequence_id", aggfunc="count", fill_value=0)

    class_matrix.to_csv(SUMMARY_DIR / "deeparg_ARG_class_count_matrix.tsv", sep="\t")

    presence_matrix = (class_matrix > 0).astype(int)
    presence_matrix.to_csv(SUMMARY_DIR / "deeparg_ARG_class_presence_matrix.tsv", sep="\t")
else:
    class_matrix = pd.DataFrame()
    presence_matrix = pd.DataFrame()

In [18]:
# Dataset-level story table
dataset_story = {
    "n_samples_scanned": len(summary_df),
    "n_samples_with_ARGs": int(summary_df["has_ARG"].sum()) if not summary_df.empty else 0,
    "n_samples_without_ARGs": int((~summary_df["has_ARG"]).sum()) if not summary_df.empty else 0,
    "total_ARG_hits": int(summary_df["n_ARGs"].sum()) if not summary_df.empty else 0,
    "total_potential_ARG_hits": int(summary_df["n_potential_ARGs"].sum()) if not summary_df.empty else 0,
    "mean_ARGs_per_sample": float(summary_df["n_ARGs"].mean()) if not summary_df.empty else 0,
    "median_ARGs_per_sample": float(summary_df["n_ARGs"].median()) if not summary_df.empty else 0,
    "max_ARGs_in_one_sample": int(summary_df["n_ARGs"].max()) if not summary_df.empty else 0,
}

pd.DataFrame([dataset_story]).to_csv(SUMMARY_DIR / "deeparg_dataset_story.tsv", sep="\t", index=False)

In [19]:
# Positive file list
positive_files = summary_df[summary_df["has_ARG"]].copy()
positive_files.to_csv(SUMMARY_DIR / "deeparg_files_with_ARGs.tsv", sep="\t", index=False)

In [37]:
dfs = {os.path.join(SUMMARY_DIR, f): pd.read_csv(os.path.join(SUMMARY_DIR, f), sep="\t") for f in os.listdir(SUMMARY_DIR)}

In [42]:
list(dfs.values())[0].T

,0
n_samples_scanned,63.000000
n_samples_with_ARGs,29.000000
n_samples_without_ARGs,34.000000
total_ARG_hits,80.000000
total_potential_ARG_hits,142.000000
mean_ARGs_per_sample,1.269841
median_ARGs_per_sample,0.000000
max_ARGs_in_one_sample,8.000000


In [51]:
for i, (path, df) in enumerate(dfs.items()):
    if i == 0:
        continue
    print(f"File: {os.path.basename(path)}")
    print(f"Shape: {df.shape}")
    if i == 2:  # For the class count matrix, show the full table
        print("non zero entries:", df[df.n_ARGs > 0].shape[0])
        print("non zero potential:", df[df.n_potential_ARGs > 0].shape[0])
        print("non zero non overlapping potential only files:", df[(df.n_ARGs == 0) & (df.n_potential_ARGs > 0)].shape[0])
    # print(df.info())
    display(df.head())

    print("\n\n")

File: deeparg_files_with_ARGs.tsv
Shape: (29, 10)


,sample,has_ARG,n_ARGs,n_potential_ARGs,n_ARG_classes,top_ARG_class,top_ARG_class_count,arg_file,potential_arg_file_exists,align_file_exists
0,bin.1,True,1,2,1,bacitracin,1,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,True,True
1,bin.10,True,6,3,5,multidrug,2,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,True,True
2,bin.14,True,7,16,4,multidrug,4,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,True,True
3,bin.16,True,1,2,1,multidrug,1,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,True,True
4,bin.19,True,1,2,1,MLS,1,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,True,True





File: deeparg_file_summary.tsv
Shape: (63, 10)
non zero entries: 29
non zero potential: 44
non zero non overlapping potential only files: 20


,sample,has_ARG,n_ARGs,n_potential_ARGs,n_ARG_classes,top_ARG_class,top_ARG_class_count,arg_file,potential_arg_file_exists,align_file_exists
0,bin.1,True,1,2,1,bacitracin,1,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,True,True
1,bin.10,True,6,3,5,multidrug,2,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,True,True
2,bin.11,False,0,0,0,NaN,0,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,True,True
3,bin.12,False,0,2,0,NaN,0,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,True,True
4,bin.13,False,0,2,0,NaN,0,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,True,True





File: deeparg_ARG_class_presence_matrix.tsv
Shape: (29, 10)


,sample,MLS,aminoglycoside,bacitracin,fosmidomycin,multidrug,peptide,phenicol,tetracycline,unclassified
0,bin.1,0,0,1,0,0,0,0,0,0
1,bin.10,0,1,1,1,1,1,0,0,0
2,bin.14,0,1,1,0,1,0,0,0,1
3,bin.16,0,0,0,0,1,0,0,0,0
4,bin.19,1,0,0,0,0,0,0,0,0





File: deeparg_all_positive_ARGs.tsv
Shape: (80, 18)


,#ARG,query-start,query-end,read_id,predicted_ARG-class,best-hit,probability,identity,alignment-length,alignment-bitscore,alignment-evalue,counts,sample,arg_file,deeparg_label,sequence_id,deeparg_best_hit,deeparg_score
0,BACA,1,291,AAHDPPLL_01139,bacitracin,YP_584847|FEATURES|bacA|bacitracin|bacA,0.992063,67.5,295,388.0,2.090000e-137,1,bin.1,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,bacitracin,AAHDPPLL_01139,YP_584847|FEATURES|bacA|bacitracin|bacA,0.992063
1,ROSB,1,552,JHMMIFGA_00180,fosmidomycin,gi:896190680:ref:WP_049199932.1:|FEATURES|rosB...,0.999900,53.4,560,528.0,5.010000e-184,1,bin.10,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,fosmidomycin,JHMMIFGA_00180,gi:896190680:ref:WP_049199932.1:|FEATURES|rosB...,0.999900
2,BACA,1,276,JHMMIFGA_01068,bacitracin,ZP_04577926|FEATURES|bacA|bacitracin|bacA,0.995254,69.7,277,374.0,2.160000e-132,1,bin.10,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,bacitracin,JHMMIFGA_01068,ZP_04577926|FEATURES|bacA|bacitracin|bacA,0.995254
3,MEXE,40,374,JHMMIFGA_01080,multidrug,YP_406903|FEATURES|mexE|multidrug|mexE,0.999998,51.2,338,300.0,2.570000e-99,1,bin.10,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,multidrug,JHMMIFGA_01080,YP_406903|FEATURES|mexE|multidrug|mexE,0.999998
4,ACRB,1,1032,JHMMIFGA_01081,multidrug,gi:503341948:ref:WP_013576609.1:|FEATURES|acrB...,0.952842,55.5,1032,1080.0,0.000000e+00,1,bin.10,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,multidrug,JHMMIFGA_01081,gi:503341948:ref:WP_013576609.1:|FEATURES|acrB...,0.952842





File: deeparg_ARG_class_count_matrix.tsv
Shape: (29, 10)


,sample,MLS,aminoglycoside,bacitracin,fosmidomycin,multidrug,peptide,phenicol,tetracycline,unclassified
0,bin.1,0,0,1,0,0,0,0,0,0
1,bin.10,0,1,1,1,2,1,0,0,0
2,bin.14,0,1,1,0,4,0,0,0,1
3,bin.16,0,0,0,0,1,0,0,0,0
4,bin.19,1,0,0,0,0,0,0,0,0


In [ ]:
arg_df = dfs["deeparg_all_positive_ARGs.tsv"]

KeyError: 'deeparg_all_positive_ARGs.tsv'

In [ ]:
arg_df = dfs["/home/ubuntu/projects/biodata/DNA-BERT/data/interim/Wetland Metagenome Assembled Genomes/_summary/deeparg_all_positive_ARGs.tsv"]

In [59]:
arg_df.head()

,#ARG,query-start,query-end,read_id,predicted_ARG-class,best-hit,probability,identity,alignment-length,alignment-bitscore,alignment-evalue,counts,sample,arg_file,deeparg_label,sequence_id,deeparg_best_hit,deeparg_score
0,BACA,1,291,AAHDPPLL_01139,bacitracin,YP_584847|FEATURES|bacA|bacitracin|bacA,0.992063,67.5,295,388.0,2.090000e-137,1,bin.1,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,bacitracin,AAHDPPLL_01139,YP_584847|FEATURES|bacA|bacitracin|bacA,0.992063
1,ROSB,1,552,JHMMIFGA_00180,fosmidomycin,gi:896190680:ref:WP_049199932.1:|FEATURES|rosB...,0.999900,53.4,560,528.0,5.010000e-184,1,bin.10,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,fosmidomycin,JHMMIFGA_00180,gi:896190680:ref:WP_049199932.1:|FEATURES|rosB...,0.999900
2,BACA,1,276,JHMMIFGA_01068,bacitracin,ZP_04577926|FEATURES|bacA|bacitracin|bacA,0.995254,69.7,277,374.0,2.160000e-132,1,bin.10,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,bacitracin,JHMMIFGA_01068,ZP_04577926|FEATURES|bacA|bacitracin|bacA,0.995254
3,MEXE,40,374,JHMMIFGA_01080,multidrug,YP_406903|FEATURES|mexE|multidrug|mexE,0.999998,51.2,338,300.0,2.570000e-99,1,bin.10,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,multidrug,JHMMIFGA_01080,YP_406903|FEATURES|mexE|multidrug|mexE,0.999998
4,ACRB,1,1032,JHMMIFGA_01081,multidrug,gi:503341948:ref:WP_013576609.1:|FEATURES|acrB...,0.952842,55.5,1032,1080.0,0.000000e+00,1,bin.10,/home/ubuntu/projects/biodata/DNA-BERT/data/in...,multidrug,JHMMIFGA_01081,gi:503341948:ref:WP_013576609.1:|FEATURES|acrB...,0.952842
